In [4]:
import os

current_dir = os.getcwd()
parent_dir = os.path.dirname(current_dir)

import pandas as pd

# Function to read in a trained model and calculate embeddings based on coordinates

In [ ]:
import torch
import sys
import pandas as pd
import os
import numpy as np
from tqdm import tqdm

sys.path.append(parent_dir)

from load_lightweight import get_mvloc_encoder

def add_embeddings_to_dataframe(df, model_path, save_csv=False, csv_path=None, chunk_size=None):

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = get_mvloc_encoder(model_path, device=device)
    model.to(device)

    def process_chunk(chunk):
        # Extract the 'latitude' and 'longitude' columns
        lat_long_array = chunk[['latitude', 'longitude']].values

        # Convert the NumPy array to a PyTorch tensor
        lat_long_tensor = torch.tensor(lat_long_array, dtype=torch.float32)

        # Create a tensor and move it to the device
        c = lat_long_tensor.to(device) 

        with torch.no_grad():
            emb = model(c.double()).detach()

        # Convert the embeddings tensor back to a DataFrame
        emb_df = pd.DataFrame(emb.cpu().numpy(), columns=[f'emb_{i}' for i in range(emb.shape[1])])

        # Concatenate the new columns to the original dataframe
        chunk_with_embeddings = pd.concat([chunk.reset_index(drop=True), emb_df], axis=1)
        return chunk_with_embeddings

    if chunk_size:
        chunks = []
        for chunk in tqdm(np.array_split(df, len(df) // chunk_size), desc="Processing chunks"):
            chunks.append(process_chunk(chunk))
        df_with_embeddings = pd.concat(chunks, axis=0)
    else:
        df_with_embeddings = process_chunk(df)

    if save_csv:
        print(f"Saving the DataFrame with embeddings to {csv_path}")
        csv_output_path = os.path.join(os.getcwd(), csv_path)
        df_with_embeddings.to_csv(csv_output_path, index=False)

    return df_with_embeddings

# The functions expects a dataframe with two columns named 'latitude' and 'longitude'
# It returns the dataframe with additional columns for the embeddings calles emb_0, emb_1, ..., emb_n, where n is the embedding dimension
# Optionally, it can save the dataframe with embeddings to a csv file if save_csv is set to True

## Example on how to apply the embedding function to a csv of locations

In [ ]:
# Read in data set with locations stored in two columns called latitude and longitude
df_locations = pd.read_csv('./locations.csv')

# Add embeddings to the dataframe using the trained model located at './EU32_GS96_OSM32.ckpt'
df_locations_withEmbeddings = add_embeddings_to_dataframe(df_locations, 
                                           './EU32_GS96_OSM32.ckpt', 
                                           save_csv=True, 
                                           csv_path='locations_withEmbeddings.csv', chunk_size=100000)
# chuck_size can be adjusted based on available memory